In [3]:
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
import time
from tqdm import tqdm
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

def create_driver():
    options = Options()
    options.add_argument("--headless") # 내부 동작 확인을 위해 첫 실행 시에는 주석 처리 권장
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/119.0.0.0 Safari/537.36")
    return webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

def get_kakao_all_info(driver, url, max_reviews=30):
    driver.get(url)

    time.sleep(4.0)

    wait = WebDriverWait(driver, 15)

    try:
        iframe = WebDriverWait(driver, 5).until(
            EC.presence_of_element_located((By.ID, "entryIframe"))
        )
        driver.switch_to.frame(iframe)
    except:
        pass
    
    results = {
        'rating': '0.0',
        'kakao_cnt': 0,
        'blog_cnt': 0,
        'total_cnt': 0,
        'reviews': []
    }
    
    try:
        wait.until(EC.presence_of_element_located((By.CLASS_NAME, "num_star")))
        # 1. 평점 추출 (사용자가 찾은 num_star 클래스)
        try:
            results['rating'] = driver.find_element(By.CLASS_NAME, "num_star").text
        except: pass

        # 2. 리뷰 개수 추출 (이미지 구조 반영: info_num 리스트 순서대로)
        try:
            info_numbers = driver.find_elements(By.CLASS_NAME, "info_num")
            
            if len(info_numbers) >= 1:
                results['kakao_cnt'] = int("".join(filter(str.isdigit, info_numbers[0].text)))

            if len(info_numbers) >= 2:
                results['blog_cnt'] = int("".join(filter(str.isdigit, info_numbers[1].text)))

            results['total_cnt'] = results['kakao_cnt'] + results['blog_cnt']
        except:
            pass

        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(2.5)

        try:
            wait.until(EC.presence_of_element_located((By.CLASS_NAME, "desc_review")))
        except:
            print("리뷰 없음 or 로딩 실패")
            return results
        
        # 3. 리뷰 텍스트 수집 (최대 max_reviews 개)
        collected_reviews = []

        while len(collected_reviews) < max_reviews:
            # --- [여기서부터 추가] 각 리뷰의 '더보기' 버튼 클릭 (내용 펼치기) ---
            while True:
                expand_btns = driver.find_elements(By.CLASS_NAME, "btn_more")
                clicked = False

                for i in range(len(expand_btns)):
                    try:
                        btn = driver.find_elements(By.CLASS_NAME, "btn_more")[i]

                        driver.execute_script("arguments[0].scrollIntoView(true);", btn)

                        if btn.text.strip() == "더보기":
                            driver.execute_script("arguments[0].click();", btn)
                            time.sleep(2.5)
                            clicked = True
                            break  # 👉 하나 클릭하면 다시 리스트 갱신
                    except:
                        continue

                if not clicked:
                    break
            # --- [여기까지 추가] ---

            # 펼쳐진 상태에서 텍스트 수집
            elements = driver.find_elements(By.CLASS_NAME, "desc_review")

            for el in elements:
                txt = el.text.strip().replace("\n", " ")
                txt = txt.replace("접기", "").strip()

                if txt and txt not in collected_reviews:
                    collected_reviews.append(txt)

                if len(collected_reviews) >= max_reviews: 
                    break
            
            if len(collected_reviews) >= max_reviews: 
                break

            # '후기 더보기' 버튼 클릭 로직 (다음 페이지 리뷰 불러오기)
            try:
                more_btns = driver.find_elements(By.CLASS_NAME, "link_more")

                clicked = False
                for btn in more_btns:
                    if "후기 더보기" in btn.text:
                        driver.execute_script("arguments[0].scrollIntoView(true);", btn)
                        driver.execute_script("arguments[0].click();", btn)
                        time.sleep(1)
                        clicked = True
                        break
                
                if not clicked:
                    break
            except:
                break
            
        results['reviews'] = collected_reviews

    except Exception as e:
        print(f"상세 데이터 수집 중 오류: {e}")

    driver.switch_to.default_content()    
    return results

# --- 메인 실행부 (5개 테스트) ---
# 1. 이전 단계에서 생성한 파일 로드
df = pd.read_csv('jeju_fail_with_url.csv',encoding='utf-8-sig')

driver = create_driver()
final_list = []

print(f"총 {len(df)}개 항목 고도화 크롤링 시작...")

for idx, row in tqdm(df.iterrows(), total=len(df)):
    if idx > 0 and idx % 150 == 0:
        driver.quit()
        time.sleep(2)
        driver = create_driver()
        
    data = get_kakao_all_info(driver, row['place_url'], max_reviews=30)
    
    final_list.append({
        'name': row['name'],
        'category': row['category'],
        'rating': data['rating'],
        'kakao_cnt': data['kakao_cnt'],
        'blog_cnt': data['blog_cnt'],
        'total_cnt': data['total_cnt'],
        'reviews_text': " | ".join(data['reviews'])
    }) 

driver.quit()

# 2. 결과 저장
result_df = pd.DataFrame(final_list)
result_df.to_csv('jeju_crawling2.csv', index=False, encoding='utf-8-sig')

print("\n--- 테스트 결과 요약 ---")

총 51개 항목 고도화 크롤링 시작...


  2%|▏         | 1/51 [00:28<23:21, 28.03s/it]

리뷰 없음 or 로딩 실패


 33%|███▎      | 17/51 [11:19<23:57, 42.27s/it]

리뷰 없음 or 로딩 실패


...

리뷰 없음 or 로딩 실패


 82%|████████▏ | 42/51 [29:40<05:29, 36.61s/it]

리뷰 없음 or 로딩 실패


100%|██████████| 51/51 [36:00<00:00, 42.36s/it]



--- 테스트 결과 요약 ---
